# Comprehensive Operational Analysis: Dustinia Delixia Groceria
**Author:** Operational Analyst  
**Objective:** To demonstrate the critical necessity of our Data Engineering ETL pipeline by first analyzing the raw, disconnected data state, and subsequently showcasing the advanced operational insights unlocked by our enriched ClickHouse Data Warehouse.

---
## Phase 1: Setup & Initialization
We begin by importing the necessary analytical and visualization libraries. We will also set a professional, clean aesthetic for our visual storytelling.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import clickhouse_connect
import warnings

# Suppress minor warnings for a clean presentation
warnings.filterwarnings('ignore')

# Set professional visualization theme
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'axes.titlesize': 16,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10
})

print("Libraries imported and visualization theme configured.")


---
## Phase 2: Raw Data Storytelling & The ETL Mandate (The "Before")
Before our Airflow DAG cleanses and structures the data, it exists as disconnected, raw CSV files on our local filesystem. Analyzing this raw data reveals massive operational blind spots and directly justifies the architecture of our Data Warehouse.

### 1. The Operational Funnel (Order Status)
Let's examine the raw distribution of order statuses. 

**The ETL Mandate:** Notice how the vast majority of orders are 'delivered'. However, a significant chunk are 'canceled', 'unavailable', or 'shipped' (in transit). In our raw state, these statuses are mixed together. This raw distribution dictates our ETL filtering logic: we must carefully isolate 'delivered' orders to calculate true historical lead times, while continuing to track 'canceled' or 'unavailable' orders in a separate dimension to identify fulfillment failures.


In [ ]:
# Load raw datasets from the local Read-Only volume mounted by Docker
raw_dir = '/home/jovyan/work/raw_data/'

# Load main tables
orders_raw = pd.read_csv(raw_dir + 'orders.csv')
customers_raw = pd.read_csv(raw_dir + 'customers.csv')
sellers_raw = pd.read_csv(raw_dir + 'sellers.csv')
reviews_raw = pd.read_csv(raw_dir + 'order_reviews.csv')
items_raw = pd.read_csv(raw_dir + 'order_items.csv')
geo_raw = pd.read_csv(raw_dir + 'geolocation.csv')

print(f"Successfully loaded {len(orders_raw):,} raw orders.")

# Calculate status distribution
status_counts = orders_raw['order_status'].value_counts()

# Plotting the Operational Funnel
plt.figure(figsize=(10, 6))
sns.barplot(y=status_counts.index, x=status_counts.values, palette="viridis")
plt.title("Raw Order Status Distribution (The Operational Funnel)")
plt.xlabel("Number of Orders")
plt.ylabel("Order Status")
plt.xscale('log') # Log scale to visualize the smaller drop-offs
plt.tight_layout()
plt.show()


### 2. The Timestamp Disconnect
To understand logistical efficiency, we must look at timelines. Let's take a random sample of 100 delivered orders and plot their journey from purchase to customer delivery.

**The ETL Mandate:** Looking at this timeline, you can visually see massive, unpredictable gaps between when a payment is approved and when the carrier actually dispatches the item. The raw timestamps alone are useless for leadership—they cannot easily aggregate or average "timestamps." This explicitly justifies why our Airflow DAG engineers the critical `processing_time_days`, `transit_time_days`, and `lead_time_days` metrics before loading the data into ClickHouse.


In [ ]:
# Filter for delivered orders and sample 100
delivered_sample = orders_raw[orders_raw['order_status'] == 'delivered'].dropna().sample(100, random_state=42)

# Convert strings to datetime
time_cols = ['order_purchase_timestamp', 'order_approved_at', 
             'order_delivered_carrier_date', 'order_delivered_customer_date']
for col in time_cols:
    delivered_sample[col] = pd.to_datetime(delivered_sample[col])

# Sort by purchase time to make the Gantt chart readable
delivered_sample = delivered_sample.sort_values('order_purchase_timestamp').reset_index(drop=True)

plt.figure(figsize=(12, 8))
for i, row in delivered_sample.iterrows():
    plt.plot([row['order_purchase_timestamp'], row['order_delivered_customer_date']], [i, i], color='grey', alpha=0.3)
    plt.scatter(row['order_purchase_timestamp'], i, color='blue', s=20, label='Purchased' if i==0 else "")
    plt.scatter(row['order_approved_at'], i, color='green', s=20, label='Approved' if i==0 else "")
    plt.scatter(row['order_delivered_carrier_date'], i, color='orange', s=20, label='Carrier' if i==0 else "")
    plt.scatter(row['order_delivered_customer_date'], i, color='red', s=20, label='Delivered' if i==0 else "")

plt.title("Order Lifecycle: The Timestamp Disconnect (Sample of 100)")
plt.xlabel("Date")
plt.ylabel("Order Index")
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()


### 3. Geospatial & Relational Complexity
The raw geospatial data (`geolocation.csv`) contains multiple overlapping zip code coordinates, and seller locations are completely detached from order items.

**The ETL Mandate:** If an analyst attempts to find regional bottlenecks using the raw CSVs, they must perform a 5-way join across millions of rows, which is computationally devastating. This justifies our decision to build a highly denormalized Star Schema (`dim_sellers`, `dim_customers`) in ClickHouse, allowing sub-second aggregations by state without complex joins.


In [ ]:
print(f"Total rows in raw geolocation table: {len(geo_raw):,}")
print(f"Unique zip codes in geolocation table: {geo_raw['geolocation_zip_code_prefix'].nunique():,}")
print(f"Duplication ratio: {len(geo_raw) / geo_raw['geolocation_zip_code_prefix'].nunique():.2f} coordinates per zip code on average.")


### 4. Data Integrity & Anomalies
Raw data is notoriously dirty. Let's look for missing values across our critical operational dates.

**The ETL Mandate:** The heatmap below highlights a critical issue: missing approval dates, missing carrier dates, and empty delivery estimates. Furthermore, relying on Pandas to handle these missing values often results in accidental float coercions or `NaT` errors. These anomalies are the final justification for our Airflow ETL quality checks and our decision to aggressively convert missing values into strict `NULL`s before pushing to ClickHouse.


In [ ]:
# Plot missing values for critical timestamp columns
plt.figure(figsize=(10, 4))
sns.heatmap(orders_raw[['order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date']].isnull(), 
            cbar=False, cmap='YlGnBu', yticklabels=False)
plt.title("Missing Value Heatmap: Critical Operational Dates")
plt.show()

# Detect illogical timelines (Delivery before Purchase)
invalid_timelines = delivered_sample[delivered_sample['order_delivered_customer_date'] < delivered_sample['order_purchase_timestamp']]
print(f"Illogical timelines detected in sample: {len(invalid_timelines)}")


---
## Phase 3: Connecting to the Source of Truth
We now transition from the messy, raw CSVs to our highly optimized ClickHouse Data Warehouse. Our ETL pipeline has already cleaned the data, calculated lead times, embedded ML predictions, and properly typed all columns.

We connect using the `jupyter_reader` account, which enforces a strict Zero-Trust read-only policy.


In [ ]:
import os

# Connect to the ClickHouse container securely
client = clickhouse_connect.get_client(
    host='clickhouse', 
    port=8123, 
    username='jupyter_reader', 
    password='clickhouse123',
    database='operational_db'
)

print("Successfully connected to the ClickHouse Data Warehouse!")


---
## Phase 4: Enriched Operational Analytics (The "After")
Now that we are connected to ClickHouse, we can perform advanced analytics at blazing speeds using our newly engineered metrics.


In [ ]:
# 1. The Operational Bottleneck (Lead Times)
query_lead_time = """
    SELECT lead_time_days 
    FROM fact_deliveries 
    WHERE lead_time_days IS NOT NULL 
      AND lead_time_days > 0 
      AND lead_time_days < 60 -- Filter out extreme extreme outliers for visualization
"""
df_lead_time = client.query_df(query_lead_time)

plt.figure(figsize=(10, 5))
sns.histplot(df_lead_time['lead_time_days'], bins=50, kde=True, color='purple')
plt.axvline(df_lead_time['lead_time_days'].median(), color='red', linestyle='dashed', linewidth=2, label=f"Median: {df_lead_time['lead_time_days'].median():.1f} days")
plt.title("Distribution of Total Delivery Lead Times")
plt.xlabel("Days from Purchase to Customer Delivery")
plt.ylabel("Frequency")
plt.legend()
plt.show()


### Geospatial Delay Heatmap
Using our enriched `is_delayed` flag (calculated in the DAG by comparing actual delivery to estimated delivery), we can instantly identify which states suffer the most logistical failures.


In [ ]:
# 2. Geospatial Delay Heatmap
query_delays = """
    SELECT 
        c.customer_state,
        COUNT(f.order_id) as total_orders,
        SUM(f.is_delayed) as delayed_orders,
        (SUM(f.is_delayed) / COUNT(f.order_id)) * 100 as delay_percentage
    FROM fact_deliveries f
    JOIN dim_customers c ON f.customer_id = c.customer_id
    WHERE f.is_delayed IS NOT NULL AND c.customer_state IS NOT NULL
    GROUP BY c.customer_state
    HAVING total_orders > 500
    ORDER BY delay_percentage DESC
    LIMIT 10
"""
df_delays = client.query_df(query_delays)

plt.figure(figsize=(10, 5))
sns.barplot(x='delay_percentage', y='customer_state', data=df_delays, palette="Reds_r")
plt.title("Top 10 States with the Highest Percentage of Late Deliveries")
plt.xlabel("Delay Percentage (%)")
plt.ylabel("Customer State")
plt.show()


### Seller Efficiency Matrix
Who is causing the delays? Are high-volume sellers overwhelmed, or are low-volume sellers simply inefficient? We analyze the `processing_time_days` metric calculated by Airflow.


In [ ]:
# 3. Seller Efficiency Matrix
query_sellers = """
    SELECT 
        s.seller_id,
        COUNT(f.order_id) as order_volume,
        AVG(f.processing_time_days) as avg_processing_time
    FROM fact_deliveries f
    JOIN dim_sellers s ON f.seller_id = s.seller_id
    WHERE f.processing_time_days IS NOT NULL AND f.processing_time_days > 0
    GROUP BY s.seller_id
    HAVING order_volume > 50
    ORDER BY avg_processing_time DESC
    LIMIT 100
"""
df_sellers = client.query_df(query_sellers)

plt.figure(figsize=(10, 6))
sns.scatterplot(x='order_volume', y='avg_processing_time', data=df_sellers, color='darkorange', alpha=0.7, s=100)
plt.title("Seller Efficiency: Order Volume vs. Avg Processing Time (Days)")
plt.xlabel("Total Order Volume")
plt.ylabel("Average Processing Time (Days)")

# Highlight the worst offender
worst_seller = df_sellers.iloc[0]
plt.annotate('Worst Bottleneck', 
             xy=(worst_seller['order_volume'], worst_seller['avg_processing_time']),
             xytext=(worst_seller['order_volume'] + 50, worst_seller['avg_processing_time']),
             arrowprops=dict(facecolor='black', shrink=0.05))
plt.show()


### The Ripple Effect: Logistics vs. Customer Satisfaction
Finally, does logistical performance actually impact the business? We join our fact table against the `dim_reviews` table to map late deliveries against 1-5 star ratings.


In [ ]:
# 4. The Ripple Effect (Logistics vs. Satisfaction)
query_reviews = """
    SELECT 
        f.is_delayed,
        r.review_score
    FROM fact_deliveries f
    JOIN dim_reviews r ON f.order_id = r.order_id
    WHERE f.is_delayed IS NOT NULL AND r.review_score IS NOT NULL
"""
df_reviews = client.query_df(query_reviews)
df_reviews['Delivery Status'] = df_reviews['is_delayed'].map({0: 'On Time', 1: 'Delayed'})

plt.figure(figsize=(8, 5))
sns.boxplot(x='Delivery Status', y='review_score', data=df_reviews, palette=['#2ecc71', '#e74c3c'])
plt.title("The Ripple Effect: How Late Deliveries Crush Review Scores")
plt.xlabel("Logistical Performance")
plt.ylabel("Review Score (1-5 Stars)")
plt.show()


---
## Phase 5: Actionable Recommendations
Based on our enriched Data Warehouse analytics, the Operations Team presents the following immediate actions:

1. **Targeted Seller Interventions:** The Seller Efficiency Matrix reveals that a handful of mid-volume sellers take upwards of 15-20 days just to hand packages to the carrier. Operations must immediately audit the top 10 worst offenders and implement a strict SLA (e.g., 48 hours to dispatch) or risk platform suspension.
2. **Geospatial Logistical Overhaul:** Specific regional states show a drastically higher probability of late deliveries. We recommend negotiating localized carrier contracts or incentivizing sellers to stock inventory in regional fulfillment centers closer to these high-delay zones.
3. **The Financial Cost of Delays:** The Boxplot analysis definitively proves that late deliveries devastate customer review scores (dropping the median score significantly). Given the direct correlation between reviews and future revenue, the predictive ML model deployed in our DAG (`predicted_delay_probability`) should be actively used by customer service to issue proactive apologies or discount codes *before* the delayed order arrives.
